# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies

We'll first install all our required libraries.

> NOTE: If you're running this locally - please skip this step.

In [2]:
#!pip install -qU langchain langchain_openai langchain-community langgraph arxiv

## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def set_api_key_if_not_present(key_name, prompt_message):
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)

set_api_key_if_not_present("OPENAI_API_KEY", "OpenAI API Key:")

In [4]:
set_api_key_if_not_present("TAVILY_API_KEY", "TAVILY_API_KEY:")

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE6 - LangGraph - {uuid4().hex[0:8]}"
set_api_key_if_not_present("LANGCHAIN_API_KEY", "LangSmith API Key:")

In [6]:
from IPython.display import display, Markdown
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableSerializable
from typing import Union, Dict, Any

def display_markdown(message : (str, AIMessage) ):
    """Display formatted Markdown text in a Jupyter notebook.
    
    Args:
        text (str): Markdown formatted text to display
    """
    if isinstance(message, AIMessage):
        display(Markdown(message.content))
    else:
        display(Markdown(message))
        
    return message

class MarkdownDisplayRunnable(RunnableSerializable):
    """A runnable that displays content as Markdown in a Jupyter notebook.
    
    Can be used in LCEL chains to display outputs.
    """
    
    def invoke(self, input: Union[str, AIMessage, Dict[str, Any]], config=None) -> Union[str, AIMessage, Dict[str, Any]]:
        """Display the input as Markdown and pass it through."""
        if isinstance(input, dict) and "output" in input:
            display_markdown(input["output"])
            return input
        else:
            display_markdown(input)
            return input    

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/tools/arxiv)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

In [7]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
]

Just to demo what the two tools do, here's the `invoke()` for each:

In [8]:
for response in tavily_tool.invoke("What is the capital of France?"):
    print(response["url"])
    display_markdown(response["content"])


https://en.wikipedia.org/wiki/List_of_capitals_of_France


Find sources: "List of capitals of France" – news · newspapers · books · scholar · JSTOR (July 2012) (Learn how and when to remove this message)
This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944.[1]
Chronology[edit]
Tournai (before 486), current-day Belgium
Soissons (486–936)
Laon (936–987)
Paris (987–1419), the residence of the Kings of France, although they were consecrated at Reims. [...] Bordeaux (September 1914), the government was relocated from Paris to Bordeaux very briefly during World War I, when it was feared that Paris would soon fall into German hands. These fears were alleviated after the German Army was pushed back at the First Battle of the Marne.
Tours (10–13 June 1940), the city served as the temporary capital of France during World War II after the government fled Paris due to the German advance. [...] Paris (1789–1871), on 5 and 6 October 1789, a throng from Paris invaded the castle and forced the royal family to move back to Paris. The National Constituent Assembly followed the King to Paris soon afterward; Versailles lost its role of capital city.
Provisional seats of the government:
Versailles (1871), the French Third Republic established Versailles as its provisional seat of government in March 1871 after the Paris Commune took control of Paris.

https://home.adelphi.edu/~ca19535/page%204.html


Home | Spain | Sydney | San Francisco | Paris | Las Vegas | Maui
Paris facts: Paris, the capital

        of France

Paris is the capital of France,
      the largest country of Europe
      with 550 000 km2 (65 millions inhabitants).

Paris has 2.234 million inhabitants
      end 2011. She is the core of Ile de France region (12 million
      people). [...] Before Paris, the capital of France
        was Lyon
        (under the Romans). Paris first became the capital of France in
        508 under King Clovis. After centuries with no unique capital of
        France, Paris retrieved its status of capital of France under King
          Philippe Auguste, who reigned between 1180 and 1223. You
        can see remains of the Philippe August Paris walls in the
        passageway between the Louvre

          parking and Louvre Museum [...] Paris remained the capital of
        France until today, with one four year interruption. During
        German occupation (WW2 , 1940-1944), the capital of France was Vichy.

go to top

Reference:
        http://www.parisdigest.com/information/facts.htm

https://en.wikipedia.org/wiki/Paris


Paris (French pronunciation: [paʁi] ⓘ) is the capital and largest city of France. With an estimated population of 2,048,472 residents in January 2025[3] in an area of more than 105 km2 (41 sq mi),[4] Paris is the fourth-most populous city in the European Union, the ninth-most populous city in Europe and the 30th most densely populated city in the world in 2022.[5] Since the 17th century, Paris has been one of the world's major centres of finance, diplomacy, commerce, culture, fashion, and [...] As the capital of France, Paris is the seat of France's national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The President of the French Republic resides at the Élysée Palace.[122] The Prime Minister's seat is at the Hôtel Matignon.[123][124] Government ministries are located in various parts of the city, many near the Hôtel Matignon.[125] [...] Jump to content
Main menu
Search
Appearance
Donate
Create account
Log in
Personal tools
        Photograph your local culture, help Wikipedia and win!
Toggle the table of contents
Paris
279 languages
Article
Talk
Read
View source
View history
Tools
Coordinates: 48°51′24″N 2°21′8″E
From Wikipedia, the free encyclopedia
This article is about the capital city of France. For other uses, see Paris (disambiguation).
"Parisien" redirects here. For other uses, see Parisien (disambiguation).
Paris

https://www.britannica.com/place/Paris


Paris is the capital of what country?
Paris is the national capital of France.
News •
JD Vance will attend AI summit in Paris and Munich security conference in first overseas trip as VP • Feb. 4, 2025, 11:53 AM ET (AP) ...(Show more)
Film director found guilty of sexual assault in France’s first big #MeToo trial • Feb. 3, 2025, 9:20 AM ET (AP)
Gisèle Pelicot's ex-husband, imprisoned for raping and drugging her, now caught up in other cases • Jan. 30, 2025, 11:56 AM ET (AP) [...] Paris, city and capital of France, situated in the north-central part of the country. People were living on the site of the present-day city, located along the Seine River some 233 miles (375 km) upstream from the river’s mouth on the English Channel (La Manche), by about 7600 bce. The modern city has spread from the island (the Île de la Cité) and far beyond both banks of the Seine. [...] 7 of History's Most Notorious Serial Killers

Where Does the Name Europe Come From?

Inventors and Inventions of the Industrial Revolution

Causes of the Great Depression

10 Legendary Creatures from Around the World
Contents
Geography & Travel Cities & Towns Cities & Towns P-S

Paris; Eiffel Tower View of the Paris skyline from Montparnasse. (more)
Paris
national capital, France
Ask the Chatbot a Question
More Actions
Print
print Print
Please select which sections you would like to print:

https://www.britannica.com/place/France


The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the ville lumière, or “city of light,” Paris has often been remade, most famously in the mid-19th century under the command of Georges-Eugène, Baron Haussman, who was committed to Napoleon III’s vision of a modern city free of the choleric swamps and congested alleys of old, with broad avenues and a regular plan. Paris is now a sprawling [...] See article: flag of France
Audio File: Anthem of France (see article)
Head Of Government:
Prime minister: Michel Barnier
(Show more)
Capital:
Paris
(Show more)
Population:
(2025 est.) 66,607,000
(Show more)
Currency Exchange Rate:
1 USD equals 0.937 euro
(Show more)
Head Of State:
President: Emmanuel Macron
(Show more)
Form Of Government:
republic with two legislative houses (Parliament; Senate [348], National Assembly [577])
(Show more)
Official Language:
French
(Show more) [...] Are you a student?
Get a special academic rate on Britannica Premium.
Subscribe
Among France’s other major cities are Lyon, located along an ancient Rhône valley trade route linking the North Sea and the Mediterranean; Marseille, a multiethnic port on the Mediterranean founded as an entrepôt for Greek and Carthaginian traders in the 6th century bce; Nantes, an industrial centre and deepwater harbour along the Atlantic coast; and Bordeaux, located in southwestern France along the Garonne River.

In [9]:
display_markdown( tool_belt[1].invoke("What is the Koopman operator?")  )

Published: 2001-05-23
Title: On Koopman-von Neumann Waves
Authors: D. Mauro
Summary: In this paper we study the classical Hilbert space introduced by Koopman and
von Neumann in their operatorial formulation of classical mechanics. In
particular we show that the states of this Hilbert space do not spread,
differently than what happens in quantum mechanics. The role of the phases
associated to these classical "wave functions" is analyzed in details. In this
framework we also perform the analog of the two-slit experiment and compare it
with the quantum case.

Published: 2003-06-05
Title: On Koopman-von Neumann Waves II
Authors: E. Gozzi, D. Mauro
Summary: In this paper we continue the study, started in [1], of the operatorial
formulation of classical mechanics given by Koopman and von Neumann (KvN) in
the Thirties. In particular we show that the introduction of the KvN Hilbert
space of complex and square integrable "wave functions" requires an enlargement
of the set of the observables of ordinary classical mechanics. The possible
role and the meaning of these extra observables is briefly indicated in this
work. We also analyze the similarities and differences between non selective
measurements and two-slit experiments in classical and quantum mechanics.

Published: 2001-11-14
Title: Minimal Coupling in Koopman-von Neumann Theory
Authors: E. Gozzi, D. Mauro
Summary: Classical mechanics (CM), like quantum mechanics (QM), can have an
operatorial formulation. This was pioneered by Koopman and von Neumann (KvN) in
the 30's. They basically formalized, via the introduction of a classical
Hilbert space, earlier work of Liouville who had shown that the classical time
evolution can take place via an operator, nowadays known as the Liouville
operator. In this paper we study how to perform the coupling of a point
particle to a gauge field in the KvN version of CM. So we basically implement
at the classical operatorial level the analog of the minimal coupling of QM. We
show that, differently than in QM, not only the momenta but also other
variables have to be coupled to the gauge field. We also analyze in details how
the gauge invariance manifests itself in the Hilbert space of KvN and indicate
the differences with QM. As an application of the KvN method we study the
Landau problem proving that there are many more degeneracies at the classical
operatorial level than at the quantum one. As a second example we go through
the Aharonov-Bohm phenomenon showing that, at the quantum level, this
phenomenon manifests its effects on the spectrum of the quantum Hamiltonian
while at the classical level there is no effect whatsoever on the spectrum of
the Liouville operator.

'Published: 2001-05-23\nTitle: On Koopman-von Neumann Waves\nAuthors: D. Mauro\nSummary: In this paper we study the classical Hilbert space introduced by Koopman and\nvon Neumann in their operatorial formulation of classical mechanics. In\nparticular we show that the states of this Hilbert space do not spread,\ndifferently than what happens in quantum mechanics. The role of the phases\nassociated to these classical "wave functions" is analyzed in details. In this\nframework we also perform the analog of the two-slit experiment and compare it\nwith the quantum case.\n\nPublished: 2003-06-05\nTitle: On Koopman-von Neumann Waves II\nAuthors: E. Gozzi, D. Mauro\nSummary: In this paper we continue the study, started in [1], of the operatorial\nformulation of classical mechanics given by Koopman and von Neumann (KvN) in\nthe Thirties. In particular we show that the introduction of the KvN Hilbert\nspace of complex and square integrable "wave functions" requires an enlargement\nof the set of 

In [10]:
arxiv_response =tool_belt[1].invoke("What is the Koopman operator?")

str

### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [9]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [10]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

#### ❗ Answer #1:

Model relies on the tool description to identify whether a particular tool
is relevant to the user's query. For example, the tools we specified above 
have the following descriptions:

In [11]:
for tool in tool_belt:
    display_markdown(f"  - **{tool.name}**: {tool.description}")


  - **tavily_search_results_json**: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.

  - **arxiv**: A wrapper around Arxiv.org Useful for when you need to answer questions about Physics, Mathematics, Computer Science, Quantitative Biology, Quantitative Finance, Statistics, Electrical Engineering, and Economics from scientific articles on arxiv.org. Input should be a search query.

Now, importantly, the model itself does not call the tool.
Rather, the model formulates a JSON packet that contains inputs for the tool.
The function call (for each tool) has to happen in a separate LangGraph node.


## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [12]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [13]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [14]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [15]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [16]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

> So the API for `add_conditional_edges` uses a `Callable` (as above) or
> `Runnable` instead of the second node (compared to `add_edge`). This means
> that we could potentially have another LLM processing the tool call request
> and sending it to an appropriate function.

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [17]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [19]:
compiled_graph = uncompiled_graph.compile()


#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

#### ❗ Answer #2:

The default recursion limit is set to 25 super-steps (sequential executions
of nodes).
It can be changed during invocation by passing kwarg 
`config={"recursion_limit" : 5} ` or whatever number we need

https://github.com/langchain-ai/langgraph/blob/main/docs/docs/concepts/low_level.md#recursion-limit

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [21]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        for message in values["messages"]:
            message.pretty_print()
        print("\n\n")

================================== Ai Message ==================================
Tool Calls:
  tavily_search_results_json (call_86Nba2qFRYhBCWKR1L1XqQxA)
 Call ID: call_86Nba2qFRYhBCWKR1L1XqQxA
  Args:
    query: current captain of the Winnipeg Jets 2023



================================= Tool Message =================================
Name: tavily_search_results_json

[{"url": "https://www.hockey-reference.com/teams/WPG/2024.html", "content": "via Sports Logos.net\n\nAbout logos\n\n2023-24\nWinnipeg Jets\nRoster and Statistics\n\nRecord: 52-24-6 (110 points), Finished 2nd in NHL Central Division\t (Schedule and Results)\n\nCoach: Rick Bowness (52-24-6)\n\nCaptain:\nAdam Lowry\n\nPrimary Arena: Canada Life Centre\n\nGoals For: 259 (15th of 32), Goals Against: 198 (1st of 32)\n\tSRS: 0.69 (4th of 32), \n\tSOS: -0.04 (29th of 32)\n\nPlayoffs:Lost First Round (4-1) to Colorado Avalanche\n\nPreseason Odds: Stanley Cup +5000; O/U: 91.5\n\nOn this page: [...] 4 | Josh Morrissey | 28 | D | 5

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [22]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out what position they hold using Tavily!")]}

async for chunk in compiled_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():

        for msg in values["messages"]:
            msg.pretty_print()
      
        print("\n\n")

================================== Ai Message ==================================
Tool Calls:
  arxiv (call_cyvNCCbrebpeaFe6vy8xEsuc)
 Call ID: call_cyvNCCbrebpeaFe6vy8xEsuc
  Args:
    query: QLoRA



================================= Tool Message =================================
Name: arxiv

Published: 2023-05-23
Title: QLoRA: Efficient Finetuning of Quantized LLMs
Authors: Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, Luke Zettlemoyer
Summary: We present QLoRA, an efficient finetuning approach that reduces memory usage
enough to finetune a 65B parameter model on a single 48GB GPU while preserving
full 16-bit finetuning task performance. QLoRA backpropagates gradients through
a frozen, 4-bit quantized pretrained language model into Low Rank
Adapters~(LoRA). Our best model family, which we name Guanaco, outperforms all
previous openly released models on the Vicuna benchmark, reaching 99.3% of the
performance level of ChatGPT while only requiring 24 hours of finetuning on a
single GPU.

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

In [46]:
import logging
logging.basicConfig(level=logging.INFO)  # or DEBUG


In [ ]:
response = compiled_graph.invoke(inputs,stream_mode="updates", config={"verbose": True})


Here's the LangSmith trace:

![Image](./Screenshot%20from%202025-04-20%2023-24-09.png)

It demonstrates that the following sequence is executed:

1. Query is passed to the LLM
1. LLM requests QLoRA paper from arxiv
1. Response is passed to LLM
1. LLM requests, for each author, their position from Tavily
1. Response is passed back to LLM which generates the final answer.

# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [24]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain = convert_inputs | compiled_graph | parse_output | display_markdown

In [25]:
agent_chain.invoke({"question" : "What is RAG?"})

RAG stands for Retrieval-Augmented Generation. It is a technique used in natural language processing (NLP) that combines retrieval-based methods with generative models to improve the quality and accuracy of generated text. Here's how it generally works:

1. **Retrieval**: The system first retrieves relevant information from a large corpus or database. This step involves searching for documents, passages, or data that are pertinent to the input query or context.

2. **Augmentation**: The retrieved information is then used to augment the input to a generative model. This means that the generative model has access to additional context or facts that can help it produce more accurate and contextually relevant responses.

3. **Generation**: Finally, the generative model uses both the original input and the retrieved information to generate a response. This can be in the form of answering questions, completing sentences, or creating more complex text outputs.

RAG is particularly useful in scenarios where the generative model alone might not have enough information to produce a high-quality response, such as in open-domain question answering or when dealing with specialized knowledge areas. By leveraging external data sources, RAG can enhance the model's performance and provide more accurate and informative outputs.

"RAG stands for Retrieval-Augmented Generation. It is a technique used in natural language processing (NLP) that combines retrieval-based methods with generative models to improve the quality and accuracy of generated text. Here's how it generally works:\n\n1. **Retrieval**: The system first retrieves relevant information from a large corpus or database. This step involves searching for documents, passages, or data that are pertinent to the input query or context.\n\n2. **Augmentation**: The retrieved information is then used to augment the input to a generative model. This means that the generative model has access to additional context or facts that can help it produce more accurate and contextually relevant responses.\n\n3. **Generation**: Finally, the generative model uses both the original input and the retrieved information to generate a response. This can be in the form of answering questions, completing sentences, or creating more complex text outputs.\n\nRAG is particularly us

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

In [26]:
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [27]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the QLoRA Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

{'example_ids': ['68e4730b-b10a-45a6-b8b7-23f075a25c86',
  'cda75ad7-c407-430e-801a-3cd00ef4b1a7',
  'cb095afe-06c5-484d-bce6-95deeaba8100',
  'e4c93266-d04f-46bd-8c52-565fea995e43',
  '15a2c5f3-985e-44bb-8000-2bbf393c8b60',
  '16ba999a-ff0a-49ef-ac11-4d9ee791d601'],
 'count': 6}

#### ❓ Question #3:

How are the correct answers associated with the questions?

> NOTE: Feel free to indicate if this is problematic or not

#### ❗ Answer #3:

I am not sure if the question here means in a programatic or semantic sense.

In the programmatic sense, the correct answer is associated with each question
by 1-1 association between elements in inputs list of dictionaries and outputs
list of dictionaries.

In the semantic sense, the correct answer is specified by "must mention" 
criterion. This may be problematic for more complex examples, as for some
questions two valid answers do not necessarily have to contain the same 
phrase.

For example, "reduce speed" and "slow down" mean the same thing, but do not 
contain common strings.


### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains *all* specific strings.

In [28]:
from langsmith.evaluation import EvaluationResult, run_evaluator
from langsmith.schemas import Run, Example

@run_evaluator
def must_mention(run:Run, example:Example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

We could convert to lower-case both the metric and matching patterns
since capitalizaton is likely secondary (and may legitimately vary).

### Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [29]:
experiment_results = client.evaluate(
    agent_chain,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"RAG Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

View the evaluation results for experiment: 'RAG Pipeline - Evaluation - 4500-e8f624bf' at:
https://smith.langchain.com/o/d5aca770-e410-428e-97f9-497517327fbd/datasets/b40e54a0-c772-4a4c-a53c-4c1a1b0b4379/compare?selectedSessions=95fb5150-8085-4804-a154-01eea03043cf




0it [00:00, ?it/s]

In 2023, the most popular deep learning frameworks are PyTorch and TensorFlow. Both frameworks are widely used and have their own strengths and communities. PyTorch is often favored for its flexibility and ease of use, making it popular among researchers, while TensorFlow is known for its robustness and is often used in industry settings. The choice between the two often depends on specific project needs and personal preference.

The LoRA (Low-Rank Adaptation) system has made several significant improvements across different domains:

1. **Federated Learning**: In the context of federated learning, LoRA has been enhanced to improve communication efficiency. The approach, known as FLASC, applies sparsity to LoRA during communication while allowing clients to locally fine-tune the entire LoRA module. This method achieves performance comparable to dense LoRA with up to 10 times less communication. Additionally, it offers benefits in terms of heterogeneity and privacy.

2. **IoT Networks**: For IoT applications, LoRA networks have been designed to be more cost-effective and flexible. The improved software architecture of the LoRa network server, which is open-source, enhances scalability and flexibility by dividing the server into decoupled modules and using a messaging system based on streaming data.

3. **Vision Applications**: In vision tasks, LoRA has been integrated into Large Multimodal Models (LMMs) to enhance their performance. The system, VaLoRA, enables accurate and efficient vision tasks by generating LoRA adapters rich in domain-specific knowledge, thus meeting application-specific accuracy requirements. This approach reduces computational expense and latency, making it more suitable for complex vision tasks.

These improvements highlight LoRA's adaptability and efficiency in various applications, from federated learning and IoT networks to complex vision tasks.

QLoRA uses "paged optimizers" to manage memory spikes during the finetuning process. This is part of the innovations introduced by QLoRA to save memory without sacrificing performance.

A Retrieval Augmented Generation (RAG) system is a type of artificial intelligence model that combines retrieval-based and generation-based approaches to improve the quality and accuracy of generated responses. Here's how it works:

1. **Retrieval Component**: This part of the system is responsible for retrieving relevant information from a large corpus of documents or a database. It uses techniques similar to those in search engines to find documents or snippets of text that are most relevant to the input query or context.

2. **Generation Component**: Once the relevant information is retrieved, the generation component uses this information to produce a coherent and contextually appropriate response. This is typically done using a language model, such as a transformer-based model, which can generate human-like text.

3. **Integration**: The integration of these two components allows the system to generate responses that are not only fluent and contextually appropriate but also grounded in factual information. This is particularly useful in applications where accuracy and reliability are crucial, such as in question answering systems, customer support, and educational tools.

By combining retrieval and generation, RAG systems can leverage the vast amount of information available in external databases while maintaining the flexibility and creativity of generative models. This approach helps in overcoming some of the limitations of purely generative models, which might otherwise produce plausible but incorrect or nonsensical answers.

The QLoRA paper introduced a new data type called "4-bit NormalFloat (NF4)," which is designed to be information theoretically optimal for normally distributed weights. This data type is part of the innovations in QLoRA to save memory without sacrificing performance.

The QLoRA paper titled "Accurate LoRA-Finetuning Quantization of LLMs via Information Retention" was authored by Haotong Qin, Xudong Ma, Xingyu Zheng, Xiaoyang Li, Yang Zhang, Shouda Liu, Jie Luo, Xianglong Liu, and Michele Magno.

In [30]:
experiment_results

,inputs.question,outputs.output,error,reference.must_mention,feedback.must_mention,execution_time,example_id,id
0,What is the most popular deep learning framework?,"In 2023, the most popular deep learning framew...",None,"[PyTorch, TensorFlow]",True,6.584425,15a2c5f3-985e-44bb-8000-2bbf393c8b60,fbbb998e-b45f-4a23-a013-f549a61d459d
1,What significant improvements does the LoRA sy...,The LoRA (Low-Rank Adaptation) system has made...,None,"[reduce, parameters]",False,8.545639,16ba999a-ff0a-49ef-ac11-4d9ee791d601,3072eab8-abd6-488e-8ab3-b46c324b191e
2,What optimizer is used in QLoRA?,"QLoRA uses ""paged optimizers"" to manage memory...",None,"[paged, optimizer]",True,4.179015,68e4730b-b10a-45a6-b8b7-23f075a25c86,d9f36df7-7a4a-4bdf-bfa2-2587fda0d4c0
3,What is a Retrieval Augmented Generation system?,A Retrieval Augmented Generation (RAG) system ...,None,"[ground, context]",True,6.456415,cb095afe-06c5-484d-bce6-95deeaba8100,d1af4323-56ae-43f8-853a-a74a9f4a2b09
4,What data type was created in the QLoRA paper?,The QLoRA paper introduced a new data type cal...,None,"[NF4, NormalFloat]",True,4.416958,cda75ad7-c407-430e-801a-3cd00ef4b1a7,8ac86178-78eb-4c27-915d-db03405091ee
5,Who authored the QLoRA paper?,"The QLoRA paper titled ""Accurate LoRA-Finetuni...",None,"[Tim, Dettmers]",False,2.826373,e4c93266-d04f-46bd-8c52-565fea995e43,aa9e65ba-8ff2-4f16-9a50-e37517822c5d


## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [31]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

> Marko: First, we define the graph and add an LLM-calling node and a tool-calling node.

In [32]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

> Marko: Then, we add the entry point to the node "agent" so the human query
> is passed to it first.

In [33]:
graph_with_helpfulness_check.set_entry_point("agent")

> OK, this is now the conditional that defines where we're going 
> after LLM returns a response.
>


In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"
  
  if len(state["messages"]) > 10:
    return "END"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]


  prompt_template = """\
  Given an initial query and a final response, determine if the final response 
  is extremely helpful or not. Please indicate helpfulness 
  with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4")

  helpfulness_chain = prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

#### 🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

> First, we check the last message to see if the tool call is requested. 
> This is just as before.
>
> Second, we exit the invocation by going to END  if the loop repetition has reached 10.
>
> If neither of those is the case, then we use the original query and the 
> last response to check how well the response matches the original query.
> We do this by formulating a new prompt sent to gpt-4 model in which
> we ask for a binary Y/N response of "helpfulness" of the query.
>
> A helpful response exits the chain.
> Unhelpful response moves to the "continue" node.

> Now we connect the possible results of the conditional:
> "continue" will call LLM (this is the "unhelpful" case)
> "action" will call the tool belt
> "end" will terminate the invokation.

In [35]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

> Last edge routes all responses from "action" node to "agent" node.

In [36]:
graph_with_helpfulness_check.add_edge("action", "agent")

> We compile and display the graph now.

In [41]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()



> Now, let's ask the system a multi-part question and stream its response.

In [38]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        for msg in values["messages"]:
            msg.pretty_print()



================================== Ai Message ==================================
Tool Calls:
  arxiv (call_aNuejdEwTRoPBzGDD66D2iEe)
 Call ID: call_aNuejdEwTRoPBzGDD66D2iEe
  Args:
    query: LoRA machine learning
  tavily_search_results_json (call_faN1wHHTuGFjJhrYOMl4SqMK)
 Call ID: call_faN1wHHTuGFjJhrYOMl4SqMK
  Args:
    query: Tim Dettmers
  arxiv (call_KF0QkOKqaHDAAjVb18f5HA3k)
 Call ID: call_KF0QkOKqaHDAAjVb18f5HA3k
  Args:
    query: Attention mechanism machine learning
================================= Tool Message =================================
Name: arxiv

Published: 2024-10-28
Title: KD-LoRA: A Hybrid Approach to Efficient Fine-Tuning with LoRA and Knowledge Distillation
Authors: Rambod Azimi, Rishav Rishav, Marek Teichmann, Samira Ebrahimi Kahou
Summary: Large language models (LLMs) have demonstrated remarkable performance across
various downstream tasks. However, the high computational and memory
requirements of LLMs are a major bottleneck. To address this,
parameter-e

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [39]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [40]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  display_markdown(what_is_string)
  display_markdown(messages["messages"][-1].content)
  print("\n\n")

What is prompt engineering and when did it break onto the scene??

**What is Prompt Engineering?**

Prompt engineering is the process of designing and refining input prompts to effectively guide the behavior of AI models. It involves crafting precise instructions or queries in natural language to elicit the best possible output from generative AI models, such as text-to-text or text-to-image models. This practice is crucial for improving the accuracy and effectiveness of AI-generated content, whether it's for generating text, images, or other media. Prompt engineering can involve specifying a style, choice of words, grammar, providing relevant context, or describing a character for the AI to mimic. It is a skill that can be utilized by anyone using generative AI tools like ChatGPT or DALL-E, as well as by AI engineers refining large language models (LLMs).

**History of Prompt Engineering**

Prompt engineering has its roots in the early days of natural language processing (NLP) and has evolved alongside the development of AI systems. The concept gained significant attention with the release of GPT-3 by OpenAI in 2020, which showcased the potential of large-scale pretrained models. This marked a watershed moment for prompt engineering, as researchers and developers began to explore crafting effective prompts to control and guide the model's behavior. The evolution of large language models (LLMs) and the introduction of reinforcement learning with human feedback (RLHF) further advanced the field, allowing models to better follow instructions and align with human intent. Prompt engineering has since become a critical component in deploying AI systems for various applications, including chatbots, virtual assistants, content generation, translation, and summarization.

What is RAG and when did it break onto the scene??

Retrieval-Augmented Generation (RAG) is a technique that enhances generative AI models by allowing them to retrieve and incorporate new information from external sources. This approach modifies interactions with large language models (LLMs) so that they can respond to user queries with reference to a specified set of documents, supplementing information from their pre-existing training data. This allows LLMs to use domain-specific and/or updated information, which is particularly useful for tasks that require current or specialized knowledge.

The concept of RAG first gained significant attention in the AI community after the publication of a 2020 paper titled "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" by Patrick Lewis and a team at Facebook AI Research. This paper highlighted the potential of RAG to significantly improve the value of generative AI systems by integrating newly available data at query time, rather than relying solely on pre-existing training data.

Historically, the roots of RAG can be traced back to the early days of computing in the 1960s and 1970s, when information retrieval systems and question-answering systems were being developed. These systems used natural language processing to access text, initially in narrow topics. Over the years, the machine learning engines driving these systems have grown significantly, increasing their usefulness and popularity.

Recent advancements in RAG technology include the development of modular frameworks and the integration of graph technology to improve the accuracy and contextual understanding of responses. These advancements aim to overcome limitations of existing RAG models, such as the inability to incorporate real-time data after the RAG configuration stage.

Overall, RAG represents a significant evolution in the field of AI, enabling more accurate, timely, and contextually relevant responses in various applications, including chatbots, email, and text messaging.

What is fine-tuning and when did it break onto the scene??

Fine-tuning in machine learning is the process of taking a pretrained model and further training it on a smaller, targeted dataset. This approach allows the model to adapt to specific tasks or datasets, improving its performance on those tasks without needing to train a new model from scratch. Fine-tuning became more prominent with the rise of deep learning and transfer learning techniques, which leverage large pretrained models to solve specific problems efficiently.

For more detailed information, you can refer to [this article on fine-tuning](https://www.techtarget.com/searchenterpriseai/definition/fine-tuning).

What is LLM-based agents and when did it break onto the scene??

LLM-based agents, or Large Language Model-based agents, are systems that utilize large language models to perform complex tasks by combining reasoning, planning, and execution capabilities. These agents leverage the power of LLMs to understand and process complex instructions, enabling them to interact with their environment in real-time and perform tasks such as web interaction, software development, and scientific discovery.

The concept of LLM-based agents gained significant attention with the popularization of OpenAI's ChatGPT in 2022. Since then, various methods and techniques have been developed to enhance their utilization and address their limitations. These agents represent a significant advancement in AI, bridging the gap between reasoning and action, and are considered the next frontier in AI development.